## Evaluate Alex's coscientist QA on gemini 2.5 pro using arize pheonix qa evaluators

In [1]:
# %pip install pdfplumber

In [2]:
# %pip install -qq "arize-phoenix-evals" ipython matplotlib pycm scikit-learn tiktoken nest_asyncio 'httpx<0.28'
# %pip install langchain_google_vertexai google-cloud-aiplatform

In [4]:
from google.cloud.aiplatform import init as init_vertexai
PROJECT_ID = "seequent-labs-dev"
LOCATION = "us-central1"
init_vertexai(project=PROJECT_ID, location=LOCATION)

# Run Gemini 2.5-pro QA agent, evaluate with custom structured output,
# log everything as a Phoenix experiment.

In [ ]:


# --------------------------------------------------------------------
import os, re, pdfplumber, pandas as pd
from pathlib import Path
from datetime import datetime
from typing import Dict, Any

import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor
from openinference.instrumentation import using_attributes
from phoenix.experiments import run_experiment
from phoenix.experiments.evaluators.utils import create_evaluator

from langchain_google_vertexai import ChatVertexAI
from langchain.schema import SystemMessage, HumanMessage
from langchain_core.pydantic_v1 import BaseModel, Field
from opentelemetry import trace                           # for manual spans

# ------------------------  CONFIG  -------------------------------
PROJECT_ID, LOCATION = "seequent-labs-dev", "us-central1"
PROJECT_NAME         = "alex-qa-coscientist-gemini"
PARENT_PATH          = Path().cwd().parent.parent
PDF_FILE             = PARENT_PATH / "data" / "qa" / "Q and A for AI evaluation.pdf"

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006"
os.environ["PHOENIX_PROJECT_NAME"]       = PROJECT_NAME
# px.launch_app() or follow https://arize.com/docs/phoenix/self-hosting/deployment-options/docker

# -----------------------------------------------------------------

# Phoenix tracer
tracer_provider = register(project_name=PROJECT_NAME, auto_instrument=True)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = tracer_provider.get_tracer(__name__)

# LLMs
agent_llm = ChatVertexAI(model_name="gemini-2.5-pro",  temperature=0.0) #llm running the agent
eval_llm  = ChatVertexAI(model_name="gemini-2.5-flash", temperature=0.0) #llm running the eval

# ------------------  DATA  ---------------------------------------
import re
from pathlib import Path
import pandas as pd
import pdfplumber

def pdf_to_df(pdf_path: Path) -> pd.DataFrame:
    """
    Extract question-answer pairs from a PDF that uses the markers
    '--Question:' and '--Answer:'.  
    Questions without a corresponding answer are skipped.
    """
    # Read the full PDF into a single text string
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            text += page_text + "\n"

    # Normalize whitespace
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"\n+", " ", text)  # Convert all line breaks to spaces
    text = re.sub(r"\s+", " ", text)  # Collapse multiple spaces
    text = text.strip()

    # Split the text into sections starting with --Question:
    question_sections = re.split(r"--Question:\s*", text)[1:]  # Skip empty first element
    
    pairs = []
    
    for section in question_sections:
        section = section.strip()
        
        # Check if this section contains an answer
        if "--Answer" in section:
            # Split at the first occurrence of --Answer (with or without colon)
            answer_match = re.search(r"--Answer\s*:?\s*", section)
            if answer_match:
                question_part = section[:answer_match.start()].strip()
                answer_part = section[answer_match.end():].strip()
                
                # Clean up the question part - remove any trailing answer content
                question_part = re.sub(r"\s*--Answer.*$", "", question_part).strip()
                
                # Clean up the answer part - remove any next question
                answer_part = re.sub(r"\s*--Question:.*$", "", answer_part).strip()
                
                # Only add if both question and answer exist and are non-empty
                if question_part and answer_part:
                    pairs.append((question_part, answer_part))
        # Skip sections without answers (as requested)
    
    return pd.DataFrame(pairs, columns=["input", "expected"])




# -----------------------------------------------------------------

# ------------------  AGENT TASK  ---------------------------------
def agent_task(output: str=None, expected:str=None, input:str="", metadata:dict|None=None) -> str:  # noqa
    """Phoenix task signature is unconstrained, but we ignore params & use internal data."""
    question=input['input']

    print(f"input: {question}\n")
    # print(f"expected: {expected}\n")
    # print(f"output: {output}\n")

    system   = "You are a helpful geostatistical assistant. Answer clearly (≤300 words), plain text. Mention any potential problems or solutions as needed."

    msgs     = [SystemMessage(content=system), HumanMessage(content=question)]

    with tracer.start_as_current_span("qa_task", kind=trace.SpanKind.CLIENT) as span:
        span.set_attribute("input", question)
        answer = agent_llm.invoke(msgs).content
        span.set_attribute("output", answer)
        return answer
# -----------------------------------------------------------------

# ------------------  CUSTOM Pydantic SCHEMA  ---------------------

# Define Pydantic Schema for Evaluation Response
class EvalResp(BaseModel):
    """Schema for evaluation response with correct/incorrect classification and explanation."""
    
    correctness: str = Field(
        description="Either 'CORRECT' if the AI answer matches the reference answer, or 'INCORRECT' if it doesn't match or is wrong"
    )
    explanation: str = Field(
        description="Brief explanation of why the AI answer is correct or incorrect compared to the reference answer"
    )


# ------------------  EVALUATOR  ----------------------------------

def map_correctness(correctness_str: str) -> int:
    """Map correctness string to binary value.
    This is a pre-requsite for the evaluator to work (i.e. the evaluator returns have to be int or float)"""
    mapper = {'CORRECT': 1, 'INCORRECT': 0}
    return mapper.get(correctness_str.upper(), 0)  # Default to 0 if not found

@create_evaluator(name="structured_correctness")
def structured_eval(output: str, expected: str, input: str = "", metadata: dict|None = None) -> Dict[str, Any]:
    """Phoenix-compliant evaluator: returns dict with keys => dataframe cols."""

    question=input['input'] # inputs are a dict, so we access the 'input' key
    expected=expected['expected'] # expected is a dict with 'expected' key, so we access it
    print(f"input: {question}\n")
    print(f"expected: {expected}\n")
    print(f"output: {output}\n")

    prompt = (
        f"You are an expert evaluator.\n\n"
        f"Question: {question}\n\n"
        f"Reference Answer: {expected}\n\n"
        f"AI Answer: {output}\n\n"
        """Compare the AI answer to the human ground truth answer, if the AI correctly answers the question,
        then the AI answer is "CORRECT". If the AI answer is longer but contains the main concepts of the
        Human answer please answer "CORRECT". If the AI answer diverges or does not contain the main
        concepts of the human answer, please answer "INCORRECT". If some concepts are missing, answer "INCORRECT".
        Terminology can be different slightly for the answer to be correct; if it is different to a degree that
        it alters the meaning, please answer "INCORRECT". If the AI answer is not relevant to the question, please answer "INCORRECT".
        Respond JSON with keys correctness ('CORRECT'/'INCORRECT') and explanation of your answer (brief)."""

    )
    structured_client = eval_llm.with_structured_output(EvalResp)
    resp: EvalResp = structured_client.invoke([HumanMessage(content=prompt)])


    return (                                # expected eval return is either an int/float value or a tuple of (float/int, string), where string is an explanation of the score
        map_correctness(resp.correctness),  
        resp.explanation,
    )



def safe_upload_dataset(dataframe, dataset_name, input_keys, output_keys):
    """
    Check if dataset exists first, then upload only if needed.
    """
    phoenix_client = px.Client()
    
    # Check existing datasets
    try:
        existing_dataset = phoenix_client.get_dataset(name=dataset_name)
        if existing_dataset:
            print(f"📋 Dataset '{dataset_name}' already exists - using existing version v{existing_dataset.version_id[:8]}")
            return existing_dataset
    except ValueError as e:
        # Dataset doesn't exist - proceed to upload
        print(f"📋 Dataset '{dataset_name}' does not exist - creating new dataset.")
        try:
            with using_attributes(
                metadata={
                    "description": "Upload new dataset to Phoenix",
                }
            ):
                exp_dataset = phoenix_client.upload_dataset(
                    dataframe=dataframe,
                    dataset_name=dataset_name,
                    input_keys=input_keys,
                    output_keys=output_keys,
                )
                print(f"📦 Uploaded new dataset v{exp_dataset.version_id[:8]} with {len(dataframe)} rows.")
                return exp_dataset
        except:
                print(f"📋 Error. Dataset '{dataset_name}' was created by another process")

    except Exception as e:
        print(f"❌ Unexpected error checking dataset: {e}")
        return None

# Usage
# Clean up the PDF file, convert into pandas df
raw_df = pdf_to_df(PDF_FILE)

# Ensure the DataFrame has the correct columns
exp_dataset = safe_upload_dataset(
    dataframe=raw_df,
    dataset_name=f"{PROJECT_NAME}_ds",
    input_keys=["input"],
    output_keys=["expected"]
)

# ------------------  RUN EXPERIMENT  -----------------------------
exp_name = f"{PROJECT_NAME}_exp_{datetime.utcnow():%Y%m%d_%H%M%S}"

exp_results = run_experiment(
    dataset           = exp_dataset,
    task              = agent_task,
    evaluators        = [structured_eval],
    experiment_name   = exp_name,
    experiment_description="Gemini 2.5-pro QA agent with custom correctness evaluator",
    concurrency       = 5,
)
print(f"✅ Experiment '{exp_name}' finished. View in Phoenix UI.")



Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: alex-qa-coscientist-gemini
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
/var/folders/z8/r7gby1gs0ts1089jpzfjwfjc0000gp/T/ipykernel_17330/1534695273.py:223: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  exp_name = f"{PROJECT_NAME}_exp_{datetime.utcnow():%Y%m%d_%H%M%S}"
🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


📋 Dataset 'alex-qa-coscientist-gemini_ds' already exists - using existing version vRGF0YXNl
🧪 Experiment started.
📺 View dataset experiments: http://localhost:6006/datasets/RGF0YXNldDox/experiments
🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDox/compare?experimentId=RXhwZXJpbWVudDo1


running tasks |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

...

input: Is Seequent using GSLIB to do geostatistics.

expected: No. Seequent uses it own algorithms that has been specifically designed to run on the cloud.

output: No, Seequent does not use GSLIB as the computational engine for its geostatistical tools, such as those found in Leapfrog Geo.

Seequent has developed its own proprietary algorithms and high-performance processing engines. For example, Leapfrog's core interpolation functionality is built on its FastRBF™ engine, which is distinct from the methods used in GSLIB.

However, Seequent's software is designed to be compatible with the wider geoscience ecosystem. It acknowledges GSLIB's importance as an industry-standard format and library. Therefore, you can:

*   **Import and export data** in GSLIB's standard file formats.
*   **Compare results** between Seequent's tools and GSLIB to validate workflows.

In summary, Seequent uses its own proprietary engine for geostatistics but ensures interoperability with GSLIB, allowing users t

/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
running experiment evaluations |█████████▏| 12/13 (92.3%) | ⏳ 01:14<00:05 |  5.30s/it

input: What is change of support and why is it important?

expected: Change of support in geostatistics refers to the process of translating spatial data or predictions from one scale to another. For instance, if you have detailed data collected at smaller spatial units (e.g., borehole samples) and you want to predict or analyze this information across larger areas or blocks (e.g., mining blocks), you need to change the support. This concept is crucial because the variability and statistical properties of the data often differ depending on the scale of measurement, and accurate modeling requires adjustments to account for these differences. Methods such as upscaling or downscaling are used to manage this transition effectively. The change of support will change the shape and characteristics of a distribution as well as the shape of the variogram. For a linear unweighted upscaling, that is averaging, the mean of the upscaled distribution will remains the same and the variance will decre

/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
running experiment evaluations |██████████| 13/13 (100.0%) | ⏳ 01:21<00:00 |  6.23s/it


🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDox/compare?experimentId=RXhwZXJpbWVudDo1

Experiment Summary (07/18/25 11:53 AM -0600)
--------------------------------------------
                evaluator   n  n_scores  avg_score
0  structured_correctness  13        13   0.769231

Tasks Summary (07/18/25 11:51 AM -0600)
---------------------------------------
   n_examples  n_runs  n_errors
0          13      13         0
✅ Experiment 'alex-qa-coscientist-gemini_exp_20250718_174750' finished. View in Phoenix UI.


In [ ]:
exp_results.as_dataframe()

,output,input,expected,example_id
run_id,,,,
RXhwZXJpbWVudFJ1bjoyMzg=,Computing statistics from drillhole assays is ...,{'input': 'How can I compute statistics from d...,{'expected': 'There are two challenges when do...,RGF0YXNldEV4YW1wbGU6NTQ1
RXhwZXJpbWVudFJ1bjoyMzk=,The primary challenge with resource estimation...,{'input': 'What is the challenge with resource...,{'expected': 'Gold is particularly difficult s...,RGF0YXNldEV4YW1wbGU6NTQ2


In [20]:
exp_results.as_dataframe()

,output,input,expected,example_id
run_id,,,,
RXhwZXJpbWVudFJ1bjoyMzg=,Computing statistics from drillhole assays is ...,{'input': 'How can I compute statistics from d...,{'expected': 'There are two challenges when do...,RGF0YXNldEV4YW1wbGU6NTQ1
RXhwZXJpbWVudFJ1bjoyMzk=,The primary challenge with resource estimation...,{'input': 'What is the challenge with resource...,{'expected': 'Gold is particularly difficult s...,RGF0YXNldEV4YW1wbGU6NTQ2


In [14]:
eval_df

,question,correct_answer,ai_generated_answer,metadata
0,How can I compute statistics from drillhole as...,There are two challenges when doing statistica...,Computing statistics from drillhole assays is ...,{'id': 'qa_0'}
1,What is the challenge with resource estimation...,Gold is particularly difficult since the value...,The primary challenge in resource estimation f...,{'id': 'qa_1'}
2,Why is domaining for resource estimation?\n--A...,A swath plot is a graphical tool used in geost...,A swath plot is a graphical validation tool us...,{'id': 'qa_2'}
3,What is the difference between an estimation a...,Two common geostatistical methods are kriging ...,Geostatistical estimation and simulation are t...,{'id': 'qa_3'}
4,Why do we need a variogram model to do a geost...,Kriging and simulations require variograms tha...,A variogram model is essential for geostatisti...,{'id': 'qa_4'}
5,Does inverse distance weighting needs a variog...,No. Inverse distance weighting (IDW) is a simp...,"No, Inverse Distance Weighting (IDW) does not ...",{'id': 'qa_5'}
6,Describe a typical geostatistical simulation w...,Geostatistical workflows typically start with ...,A typical geostatistical simulation workflow i...,{'id': 'qa_6'}
7,Describe a typical geostatistical simulation w...,The number of simulations depends on the objec...,There is no single correct number of geostatis...,{'id': 'qa_7'}
8,How can I characterize the geological uncertai...,The two main approaches are\n1- Create several...,Characterizing geological uncertainty involves...,{'id': 'qa_8'}
9,What is a normal score transform of the data a...,Normal score transform or Gaussian anamorphosi...,A normal score transform (NST) is a data conve...,{'id': 'qa_9'}


In [27]:
eval_df

,input,reference,output,metadata
0,How can I compute statistics from drillhole as...,There are two challenges when doing statistica...,Computing statistics from drillhole assays req...,{'id': 'qa_0'}
1,What is the challenge with resource estimation...,Gold is particularly difficult since the value...,The primary challenge in estimating gold resou...,{'id': 'qa_1'}
2,Why is domaining for resource estimation?\n--A...,A swath plot is a graphical tool used in geost...,A swath plot is a graphical validation tool us...,{'id': 'qa_2'}
3,What is the difference between an estimation a...,Two common geostatistical methods are kriging ...,Geostatistical estimation and simulation are t...,{'id': 'qa_3'}
4,Why do we need a variogram model to do a geost...,Kriging and simulations require variograms tha...,A variogram model is essential for geostatisti...,{'id': 'qa_4'}
5,Does inverse distance weighting needs a variog...,No. Inverse distance weighting (IDW) is a simp...,"No, Inverse Distance Weighting (IDW) does not ...",{'id': 'qa_5'}
6,Describe a typical geostatistical simulation w...,Geostatistical workflows typically start with ...,A typical geostatistical simulation workflow i...,{'id': 'qa_6'}
7,Describe a typical geostatistical simulation w...,The number of simulations depends on the objec...,"The number of geostatistical simulations, or r...",{'id': 'qa_7'}
8,How can I characterize the geological uncertai...,The two main approaches are\n1- Create several...,Characterizing geological uncertainty involves...,{'id': 'qa_8'}
9,What is a normal score transform of the data a...,Normal score transform or Gaussian anamorphosi...,A normal score transform (NST) is a data conve...,{'id': 'qa_9'}


In [28]:
# from phoenix.trace import SpanEvaluations
# import os

# px.Client().log_evaluations(
#     SpanEvaluations(
#         dataframe=eval_results,
#         eval_name="AIvsHuman",
#     ),
# )

In [34]:
schema_results

,input,reference,output,label,explanation,metadata
0,How can I compute statistics from drillhole as...,There are two challenges when doing statistica...,Computing statistics from drillhole assays req...,CORRECT,The AI answer correctly identifies the two mai...,{'id': 'qa_0'}
1,What is the challenge with resource estimation...,Gold is particularly difficult since the value...,The primary challenge in estimating gold resou...,CORRECT,The AI answer accurately identifies the core c...,{'id': 'qa_1'}
2,Why is domaining for resource estimation?\n--A...,A swath plot is a graphical tool used in geost...,A swath plot is a graphical validation tool us...,CORRECT,"The AI answer accurately defines a swath plot,...",{'id': 'qa_2'}
3,What is the difference between an estimation a...,Two common geostatistical methods are kriging ...,Geostatistical estimation and simulation are t...,CORRECT,The AI answer accurately explains the differen...,{'id': 'qa_3'}
4,Why do we need a variogram model to do a geost...,Kriging and simulations require variograms tha...,A variogram model is essential for geostatisti...,CORRECT,The AI-generated answer is correct and provide...,{'id': 'qa_4'}
5,Does inverse distance weighting needs a variog...,No. Inverse distance weighting (IDW) is a simp...,"No, Inverse Distance Weighting (IDW) does not ...",CORRECT,The AI answer correctly states that Inverse Di...,{'id': 'qa_5'}
6,Describe a typical geostatistical simulation w...,Geostatistical workflows typically start with ...,A typical geostatistical simulation workflow i...,CORRECT,The AI-generated answer accurately describes a...,{'id': 'qa_6'}
7,Describe a typical geostatistical simulation w...,The number of simulations depends on the objec...,"The number of geostatistical simulations, or r...",CORRECT,The AI answer correctly identifies that the nu...,{'id': 'qa_7'}
8,How can I characterize the geological uncertai...,The two main approaches are\n1- Create several...,Characterizing geological uncertainty involves...,CORRECT,The AI answer correctly identifies the two mai...,{'id': 'qa_8'}
9,What is a normal score transform of the data a...,Normal score transform or Gaussian anamorphosi...,A normal score transform (NST) is a data conve...,CORRECT,The AI answer accurately defines normal score ...,{'id': 'qa_9'}


In [36]:
eval_results

,label,explanation,exceptions,execution_status,execution_seconds
0,None,None,"[PhoenixTemplateMappingError(""Missing template...",MISSING INPUT,0.00339
1,None,None,[],DID NOT RUN,0.00000
2,None,None,[],DID NOT RUN,0.00000
3,None,None,[],DID NOT RUN,0.00000
4,None,None,[],DID NOT RUN,0.00000
5,None,None,[],DID NOT RUN,0.00000
6,None,None,[],DID NOT RUN,0.00000
7,None,None,[],DID NOT RUN,0.00000
8,None,None,[],DID NOT RUN,0.00000
9,None,None,[],DID NOT RUN,0.00000


In [75]:
# eval_df.iloc[:1,:].to_csv(parent_path / "data" / "qa" / "alex_qa_eval_results.csv", index=False)

In [12]:
from phoenix.trace import SpanEvaluations
import os

px.Client().log_evaluations(
    SpanEvaluations(
        dataframe=eval_results,
        eval_name="AIvsHuman",
    ),
)

/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(


ValueError: The dataframe index must be ['context.span_id'] but was '[None]'